IMPORTING LIBRARIES

In [24]:
import pandas as pd
import pyodbc

In [25]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=MSI\SQLEXPRESS;"
    "DATABASE=Ecommerce_Analysis;"
    "Trusted_Connection=yes;"
    )

<>:3: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
C:\Users\rahul\AppData\Local\Temp\ipykernel_8808\1085372439.py:3: SyntaxWarning: invalid escape sequence '\S'
  "SERVER=MSI\SQLEXPRESS;"


In [26]:
orders = pd.read_sql('SELECT * FROM orders', conn)
customers = pd.read_sql('SELECT * FROM customers', conn)
order_items = pd.read_sql('SELECT * FROM order_items', conn)
payments = pd.read_sql('SELECT * FROM payments', conn)
reviews = pd.read_sql('SELECT * FROM reviews', conn)
products = pd.read_sql('SELECT * FROM products', conn)

C:\Users\rahul\AppData\Local\Temp\ipykernel_8808\1572655370.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders = pd.read_sql('SELECT * FROM orders', conn)
C:\Users\rahul\AppData\Local\Temp\ipykernel_8808\1572655370.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql('SELECT * FROM customers', conn)
C:\Users\rahul\AppData\Local\Temp\ipykernel_8808\1572655370.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  order_items = pd.read_sql('SELECT * FROM order_items', conn)
C:\Users\rahul\AppData\L

CHECK STRUCTURE OF DATASET

In [27]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [28]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


DETECT MISSING VALUE

In [29]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

HANDLE MISSING VALUES

In [54]:
orders['delivered_flag'] = orders['order_delivered_customer_date'].notnull()

REMOVE DUPLICATES

In [31]:
orders.duplicated().sum()

np.int64(0)

In [32]:
# if duplicates exist drop 
orders = orders.drop_duplicates()

CONVERT DATE COLUMNS

In [33]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

CREATE ANALYTICAL COLUMNS

In [34]:
# ORDERS YEAR
orders['order_year'] = orders['order_purchase_timestamp'].dt.year

In [35]:
# ORDERS MONTH
orders['order_month'] = orders['order_purchase_timestamp'].dt.month

In [36]:
# ORDERS DAY
orders['order_day'] = orders['order_purchase_timestamp'].dt.day

CALCULATE DELIVERY TIME

In [37]:
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days

In [38]:
orders.head(100)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivered_flag,order_year,order_month,order_day,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,2017,10,2,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,True,2018,7,24,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,2018,8,8,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,True,2017,11,18,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,True,2018,2,13,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,77e9941864fc840be8e4b1ba5347c0f7,3135962ee745ef39b85576df7ddbaa99,delivered,2018-08-03 08:59:39,2018-08-03 09:31:36,2018-08-03 10:10:00,2018-08-17 00:49:41,2018-08-27,True,2018,8,3,13.0
96,41bb5cee06dbf170878a9ef93ac7e7f5,1833a0540067becaf59368fe4cd4303a,delivered,2018-05-14 08:35:33,2018-05-14 08:52:24,2018-05-16 14:46:00,2018-05-18 14:48:38,2018-06-08,True,2018,5,14,4.0
97,6a0a8bfbbe700284feb0845d95e0867f,68451b39b1314302c08c65a29f1140fc,delivered,2017-11-22 11:32:22,2017-11-22 11:46:50,2017-11-27 13:39:35,2017-12-28 19:43:00,2017-12-11,True,2017,11,22,36.0
98,f7959f8385f34c4f645327465a1c9fc4,0bf19317b1830a69e55b40710576aa7a,delivered,2017-03-30 07:50:33,2017-03-30 08:05:08,2017-03-30 10:55:54,2017-04-10 02:59:52,2017-04-26,True,2017,3,30,10.0


MERGE TABLES (CREATE FINAL DATASET)

In [51]:
ecommerce_sales = orders.merge(customers, on='customer_id')
ecommerce_sales = ecommerce_sales.merge(order_items, on='order_id')
ecommerce_sales = ecommerce_sales.merge(products, on='product_id')
ecommerce_sales = ecommerce_sales.merge(payments, on='order_id')

SAVE CLEAN DATASET

In [52]:
ecommerce_sales.to_csv('cleaned_ecommerce_dataset.csv', index=False)